# HDF5 Data Operations
Core functions for reading/writing to the HDF5 data file. This notebook is imported by the main server.

In [ ]:
# HDF5 helper functions
import time

def get_h5_file(h5py_module, data_file_path):
    """Open and return the HDF5 file handle with retry logic for Windows file locks"""
    max_retries = 5
    retry_delay = 0.5
    
    for attempt in range(max_retries):
        try:
            return h5py_module.File(str(data_file_path), "a")
        except OSError as e:
            if attempt < max_retries - 1:
                print(f"File lock error (attempt {attempt + 1}/{max_retries}), retrying in {retry_delay}s...")
                time.sleep(retry_delay)
            else:
                print(f"Failed to open HDF5 file after {max_retries} attempts.")
                print("Try running: backend/kill_processes.ps1")
                raise

In [ ]:
def get_sequencer_grid(get_h5_file_fn):
    """Read sequencer dataset and return as 2x12 grid"""
    with get_h5_file_fn() as f:
        if "sequencer" not in f.keys():
            import h5py
            f.create_dataset("sequencer", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer"]
        if len(seq.shape) == 3:
            if seq.shape[2] == 0:
                grid = [["" for _ in range(12)] for _ in range(2)]
            else:
                grid = seq[:, :, 0].tolist()
                grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else str(cell) if cell else "" for cell in row] for row in grid]
                grid = [[cell.strip() if isinstance(cell, str) and cell.strip() else "" for cell in row] for row in grid]
        else:
            grid = seq[:].tolist()
            grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else str(cell) if cell else "" for cell in row] for row in grid]
            grid = [[cell.strip() if isinstance(cell, str) and cell.strip() else "" for cell in row] for row in grid]
        return grid


def set_sequencer_grid(get_h5_file_fn, grid):
    """Write 2x12 grid to sequencer dataset"""
    with get_h5_file_fn() as f:
        if "sequencer" not in f.keys():
            import h5py
            f.create_dataset("sequencer", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer"]
        encoded_grid = []
        for row in grid:
            encoded_row = []
            for cell in row:
                if cell is None:
                    encoded_row.append(b'')
                elif isinstance(cell, bytes):
                    encoded_row.append(cell)
                elif isinstance(cell, str):
                    encoded_row.append(cell.encode('utf-8'))
                else:
                    encoded_row.append(str(cell).encode('utf-8'))
            encoded_grid.append(encoded_row)
        
        if len(seq.shape) == 3:
            if seq.shape[2] == 0:
                seq.resize((2, 12, 1))
            seq[:, :, 0] = encoded_grid
        else:
            seq[:] = encoded_grid
        f.flush()

In [ ]:
def get_effects_grid(get_h5_file_fn):
    """Read sequencer_effects dataset and return as 2x12 grid"""
    with get_h5_file_fn() as f:
        if "sequencer_effects" not in f.keys():
            import h5py
            f.create_dataset("sequencer_effects", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer_effects"]
        if len(seq.shape) == 3:
            if seq.shape[2] == 0:
                grid = [["" for _ in range(12)] for _ in range(2)]
            else:
                grid = seq[:, :, 0].tolist()
                grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        else:
            grid = seq[:].tolist()
            grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        return grid


def set_effects_grid(get_h5_file_fn, grid):
    """Write 2x12 grid to sequencer_effects dataset"""
    with get_h5_file_fn() as f:
        if "sequencer_effects" not in f.keys():
            import h5py
            f.create_dataset("sequencer_effects", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer_effects"]
        if len(seq.shape) == 3:
            if seq.shape[2] == 0:
                seq.resize((2, 12, 1))
            seq[:, :, 0] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]
        else:
            seq[:] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]

In [ ]:
def create_sequencer_effect(get_h5_file_fn, hruuid, datetime_module, effect_type: str, row: int, col: int, properties: dict = None):
    """Create a sequencer effect group in sequencer_effects_properties and return its UUID"""
    with get_h5_file_fn() as f:
        effects_props_group = f.require_group("sequencer_effects_properties")
        effect_uuid = hruuid.generate()
        sequencer_effect = effects_props_group.require_group(effect_uuid)
        sequencer_effect.attrs["effect_type"] = effect_type
        sequencer_effect.attrs["sequencer_row"] = row
        sequencer_effect.attrs["sequencer_col"] = col
        sequencer_effect.attrs["timestamp"] = datetime_module.now().isoformat()
        
        if properties:
            for key, value in properties.items():
                sequencer_effect.attrs[f"prop_{key}"] = value
        
        return effect_uuid